# Fit-Scorer Validation Study (Deliverable #2)

Presentation layer over the harness output. Generate the data first (from the repo root):

```bash
npm run validate:prep    # Datasets/ -> resumes.jsonl + field_profiles.json
npm run validate:fit     # live engine scores every resume x 21 field profiles
npm run validate:report  # metrics.json + confusion_matrix.png + accuracy_by_arm.png
```

For the real embeddings arm: `npm i @xenova/transformers` then
`EMBEDDINGS_ENABLED=true npm run validate:fit -- --sample 40`.

Targets: **top-1 ≥ 70%, top-3 ≥ 90%**.

In [ ]:
import json, os
import pandas as pd
from IPython.display import Image, display

# Works whether the notebook runs from the repo root or from scripts/validation/.
ART = next(p for p in ('scripts/validation/.artifacts', '.artifacts') if os.path.isdir(p))
m = json.load(open(os.path.join(ART, 'metrics.json')))
print(f"{m['n_resumes']} resumes, {m['n_families']} families")
pd.DataFrame([
    {'arm': a, 'top-1': f"{v['top1']:.1%}", 'top-3': f"{v['top3']:.1%}"}
    for a, v in m['arms'].items()
])

In [ ]:
for fig in ('confusion_matrix.png', 'accuracy_by_arm.png'):
    p = os.path.join(ART, fig)
    if os.path.exists(p):
        display(Image(filename=p))

In [ ]:
pf = m['per_family_structured']
(pd.DataFrame([{'family': k, 'n': v['n'], 'top-1': v['top1']} for k, v in pf.items()])
   .sort_values('top-1', ascending=False)
   .reset_index(drop=True))

## Reading the result

The **structured** arm is the live engine (`lib/matching.ts`) exactly as it ships. The
**embeddings** arm is BGE-small cosine (non-semantic mock unless `EMBEDDINGS_ENABLED=true`
was set during scoring). **combined** is a per-resume min-max blend of the two.

See `scripts/validation/README.md` for the current baseline and the levers to reach target.